In [78]:
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_community.vectorstores import Chroma
from pypdf import PdfReader

#Evaluacion 3

import logging
import json
import time
import psutil



In [79]:
#evaluacion 3

logging.basicConfig(
    filename="logs_asistente.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

metricas = {
    "consultas": 0,
    "errores": 0,
    "latencia_total": 0,
    "uso_memoria": [],
    "respuestas_exitosas": 0
}

In [80]:
#v3
class RootCauseAnalyzer:

    def analyze_incident(self, incident_log):

        capa = incident_log.get(
            "capa_afectada",
            "Desconocida"
        )

        firma = incident_log.get(
            "firma_ataque",
            "Sin firma"
        )

        if "api_key" in firma.lower():

            causa = "Intento de fuga de credenciales"

            recomendacion = (
                "Bloquear exposición de secretos"
            )

        else:

            causa = "Incidente genérico"

            recomendacion = (
                "Revisar logs del sistema"
            )

        return {
            "diagnostico": causa,
            "recomendacion": recomendacion
        }

        rca = RootCauseAnalyzer()

In [81]:
palabras_bloqueadas = [
    "ignore all",
    "api_key",
    "passwd",
    "root_access",
    "bucle infinito"
]

In [82]:
import os
from pypdf import PdfReader
from langchain_core.documents import Document

documentos = []

carpeta = "pdfs"

for archivo in os.listdir(carpeta):

    if archivo.endswith(".pdf"):

        ruta = os.path.join(carpeta, archivo)

        reader = PdfReader(ruta)

        texto = ""

        for pagina in reader.pages:

            contenido = pagina.extract_text()

            if contenido:
                texto += contenido

        documentos.append(
            Document(
                page_content=texto,
                metadata={"archivo": archivo}
            )
        )

print(f"Se cargaron {len(documentos)} documentos")

Se cargaron 5 documentos


In [83]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(
    openai_api_key=os.getenv("GITHUB_TOKEN"),
    openai_api_base=os.getenv("GITHUB_BASE_URL"),
    model="text-embedding-3-small"
)

In [84]:
!pip install -q langchain-chroma

from langchain_chroma import Chroma

db = Chroma.from_documents(
    documentos,
    embeddings
)

print("Base lista")

Base lista


In [85]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    base_url=os.getenv("GITHUB_BASE_URL"),
    api_key=os.getenv("GITHUB_TOKEN"),
    model="gpt-4o"
)

Evaluacion 2

In [86]:
chat_history = []

In [87]:
def decidir_accion(pregunta):
    pregunta = pregunta.lower()

    if "resum" in pregunta:
        return "resumir"
    elif "guardar" in pregunta:
        return "guardar"
    elif "documento" in pregunta or "buscar" in pregunta:
        return "rag"
    else:
        return "rag"

In [88]:
def resumir_texto(texto):
    prompt = f"Resume este texto:\n\n{texto}"
    respuesta = llm.invoke(prompt)
    return respuesta.content

In [89]:
def guardar_memoria(texto):
    chat_history.append(texto)
    return "Guardado en memoria corta"

In [90]:
#v3
def analizar_logs():

    print("\n--- ANALISIS DE LOGS ---")

    if metricas["errores"] > 5:
        print("Anomalia detectada: exceso de errores")

    if metricas["consultas"] > 20:
        print("Alto volumen de consultas")

    if metricas["consultas"] > 0:

        promedio = (
            metricas["latencia_total"] /
            metricas["consultas"]
        )

        print(
            f"Latencia promedio: {promedio:.2f} segundos"
        )

    print("------------------------")

In [91]:
def rag_response(pregunta):

    inicio = time.time()

    resultado = db.similarity_search(pregunta, k=3)

    contexto = ""

    for doc in resultado:
        contexto += doc.page_content + "\n"

    prompt = f"""
    Responde usando solamente la información encontrada.

    {contexto}

    Pregunta:
    {pregunta}
    """

    respuesta = llm.invoke(prompt)

    latencia = time.time() - inicio

    metricas["consultas"] += 1
    metricas["respuestas_exitosas"] += 1
    metricas["latencia_total"] += latencia

    memoria_actual = (
        psutil.Process()
        .memory_info()
        .rss / 1024 / 1024
    )

    metricas["uso_memoria"].append(memoria_actual)

    logging.info(
        f"Consulta procesada | Latencia: {latencia:.2f}s"
    )

    return respuesta.content

In [92]:
#v3
def agente(pregunta):

    try:

        print("Pensando...")

        chat_history.append(pregunta)

        accion = decidir_accion(pregunta)

        if accion == "resumir":
            return resumir_texto(pregunta)

        elif accion == "guardar":
            return guardar_memoria(pregunta)

        elif accion == "rag":
            return rag_response(pregunta)

        else:
            return "No entiendo la acción"

    except Exception as e:

        metricas["errores"] += 1

        logging.error(str(e))

        return "Error al procesar la consulta"

In [97]:
print(agente("¿Qué es el certificado de número?"))
print(agente("¿como sacar el permiso de circulacion?"))
print(agente("¿donde se paga el permiso de circulacion?"))



print(agente("guardar esta información"))

Pensando...
El Certificado de Número es un trámite que identifica una propiedad dentro de la comuna de Peñaflor, asignándole un número único dentro de la calle en la que se encuentra.
Pensando...
Para sacar el permiso de circulación, debes seguir estos pasos según la información disponible:

1. Acude presencialmente a la **Dirección de Tránsito y Transporte Público - Departamento de Permisos de Circulación**, ubicada en **Av. Alcalde Luis Araya Cereceda N°1215, Peñaflor**.

2. **Reúne los documentos requeridos** según tu tipo de vehículo:
   - Para renovación: Revisión técnica al día, seguro obligatorio y el permiso de circulación del año anterior.
   - Para vehículos nuevos: Factura de compra, inscripción en el Registro Civil, Certificado de Homologación y Póliza de Seguro.
   - Para locomoción colectiva y vehículos de carga: Factura (en caso de vehículos nuevos), Certificado de inscripción, Revisión técnica y emisión de gases, Certificado de Normas y Seguro obligatorio. Los taxis deb

In [94]:
#v3

print("Cantidad de elementos en memoria:", len(chat_history))

Cantidad de elementos en memoria: 4


In [95]:
#v3

print("Contenido de la memoria:")
print(chat_history)


Contenido de la memoria:
['¿Qué es el certificado de número?', '¿como sacar el permiso de circulacion?', 'guardar esta información', 'guardar esta información']


In [98]:
#v3

print("METRICAS DEL SISTEMA")

print("Consultas:", metricas["consultas"])
print("Errores:", metricas["errores"])

if metricas["consultas"] > 0:

    print(
        "Latencia promedio:",
        metricas["latencia_total"] /
        metricas["consultas"]
    )

if len(metricas["uso_memoria"]) > 0:

    print(
        "Memoria promedio:",
        sum(metricas["uso_memoria"])
        / len(metricas["uso_memoria"]),
        "MB"
    )

analizar_logs()

METRICAS DEL SISTEMA
Consultas: 5
Errores: 0
Latencia promedio: 3.8202892303466798
Memoria promedio: 507.8984375 MB

--- ANALISIS DE LOGS ---
Latencia promedio: 3.82 segundos
------------------------
